# CosyVoice 2 clone spike (free Colab)

Part of the Timbre clone vendor spike (`.plans/06-clone-vendor-decision.md`). Generates zero-shot clones of each consenting speaker with CosyVoice 2 (0.5B).

Before you start:

- Runtime → Change runtime type → **T4 GPU** (free tier). Don't buy compute units.
- Only use recordings from speakers who signed the spike consent, which covers Google Colab processing.
- Upload `recordings.zip`: the `recordings/` folder from the README layout (one folder per speaker with `prompt.*`, `prompt.txt` and `real.*`).
- Run the last cell when you're done. It deletes the recordings and outputs from this runtime.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install CosyVoice 2

Takes several minutes. If an install step fails, check the repository's issues for the current Colab workaround and pin the versions it suggests.

In [ ]:
!apt-get -qq install -y sox libsox-dev ffmpeg > /dev/null
!git clone --quiet --recursive https://github.com/FunAudioLLM/CosyVoice.git
%cd /content/CosyVoice
!pip install -q -r requirements.txt

In [ ]:
# Model weights from the official FunAudioLLM release. Check the licence of
# both code and weights on the repository before any production use.
from huggingface_hub import snapshot_download
snapshot_download("FunAudioLLM/CosyVoice2-0.5B", local_dir="pretrained_models/CosyVoice2-0.5B")

## 2. Upload the recordings

In [ ]:
import pathlib, subprocess, zipfile
from google.colab import files

uploaded = files.upload()  # choose recordings.zip
zip_name = next(iter(uploaded))
work = pathlib.Path("/content/spike")
work.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(work)
pathlib.Path(zip_name).unlink()

recordings = work / "recordings"
speakers = sorted(p for p in recordings.iterdir() if p.is_dir())
print("Speakers:", [s.name for s in speakers])

## 3. Generate the clips

Keep `SENTENCES` identical to `spike_config.py`, so both providers say the same lines.

In [ ]:
import json, sys, time
import torch, torchaudio

sys.path.append("third_party/Matcha-TTS")
from cosyvoice.cli.cosyvoice import CosyVoice2
from cosyvoice.utils.file_utils import load_wav

SENTENCES = [
    "Every voice has a timbre, a colour that makes it yours and no one else's.",
    "The weather should clear up in time for the weekend.",
    "I finished the book you lent me, and I liked the ending.",
]
AUDIO_EXTENSIONS = (".wav", ".m4a", ".mp3", ".webm", ".ogg", ".aac", ".flac")

model = CosyVoice2("pretrained_models/CosyVoice2-0.5B", load_jit=False, load_trt=False, fp16=False)
out_dir = work / "outputs" / "cosyvoice"
out_dir.mkdir(parents=True, exist_ok=True)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
timings = []

for folder in speakers:
    prompt = next(p for p in folder.iterdir() if p.stem == "prompt" and p.suffix.lower() in AUDIO_EXTENSIONS)
    prompt_text = (folder / "prompt.txt").read_text().strip()
    prompt_16k_path = folder / "prompt_16k.wav"
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(prompt), "-ac", "1", "-ar", "16000", str(prompt_16k_path)], check=True)
    prompt_speech = load_wav(str(prompt_16k_path), 16000)
    for index, text in enumerate(SENTENCES, start=1):
        started = time.monotonic()
        chunks = [c["tts_speech"] for c in model.inference_zero_shot(text, prompt_text, prompt_speech, stream=False)]
        speech = torch.cat(chunks, dim=1)
        seconds = time.monotonic() - started
        target = out_dir / f"{folder.name}_{index}.wav"
        torchaudio.save(str(target), speech, model.sample_rate)
        timings.append({"speaker": folder.name, "sentence": index, "synthesis_seconds": round(seconds, 2),
                        "audio_seconds": round(speech.shape[1] / model.sample_rate, 2), "gpu": gpu})
        print(f"{target.name}: {seconds:.1f} s")

(out_dir / "timings.json").write_text(json.dumps(timings, indent=2))

## 4. Download the outputs

Unzip into the spike folder so the clips land in `outputs/cosyvoice/`.

In [ ]:
import shutil
archive = shutil.make_archive("/content/cosyvoice-outputs", "zip", work / "outputs")
files.download(archive)

## 5. Clean up

Deletes the recordings, the converted prompts and the outputs from this runtime. Then use Runtime → Disconnect and delete runtime.

In [ ]:
shutil.rmtree(work, ignore_errors=True)
pathlib.Path("/content/cosyvoice-outputs.zip").unlink(missing_ok=True)
print("Deleted. Now disconnect and delete the runtime.")